# Lab 09: Groups


**Helpful Resource:**
- [Python Reference](http://data8.org/sp24/reference/): Cheat sheet of helpful array & table methods!

**Recommended Readings**:

* [Visualizing Numerical Distributions](https://www.inferentialthinking.com/chapters/07/2/Visualizing_Numerical_Distributions.html)
* [Functions and Tables](https://www.inferentialthinking.com/chapters/08/Functions_and_Tables.html)

Please complete this notebook by filling in the cells provided. **Before you begin, execute the cell below to setup the notebook by importing some helpful libraries.** Each time you start your server, you will need to execute this cell again.


In [ ]:
!apt-get install texlive texlive-xetex texlive-latex-extra pandoc
!pip install pypandoc
!pip install datascience

In [ ]:
# Connect Google Drive to Colab so you can access your files
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/Colab Notebooks/')

In [ ]:
# Run this cell to set up the notebook, but please don't change it.

# These lines import the Numpy and Datascience modules.
import numpy as np
from datascience import *

# These lines do some fancy plotting magic.
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

import warnings
warnings.simplefilter('ignore', FutureWarning)
warnings.filterwarnings("ignore")

## 1. Based on the Lecture Activity

In [ ]:
drinks = Table(['Drink', 'Cafe', 'Price'])
drinks = drinks.with_rows([
    ['Milk Tea', 'Asha', 5.5],
    ['Espresso', 'Strada',  1.75],
    ['Latte',    'Strada',  3.25],
    ['Espresso', "FSM",   2]
])
drinks

In [ ]:
discounts = Table().with_columns(
    'Coupon % off', make_array(10, 25, 5),
    'Location', make_array('Asha', 'Strada', 'Asha')
)
discounts

<br>


**Question 1** Join drinks and discounts based on Cafe/Location.

In [ ]:
combined = drinks.join("Cafe",discounts,"Location")
combined

<br>


**Question 2** Calculate the lowest rate you can get for each drink using the available coupons.

In [ ]:
discount_frac = combined.column("Coupon % off") / 100
combined.with_column("apply coupon",combined.column("Price") * (1 - discount_frac)).group("Drink", min)

<br>

**Question 3** What happens if I run the following? How many rows will it produce? Why?


In [ ]:
drinks.join('Cafe', drinks, 'Cafe')

The code drinks.join('Cafe', drinks, 'Cafe') performs a self-join of the drinks table. In a join operation, when the join key matches, every possible pair of rows from the left table and the right table is combined.
Specifically, the original table has 'Strada' appearing twice, so joining on 'Cafe' will produce 2×2=4 rows for 'Strada'. Since 'Asha' and 'FSM' each appear once, they will generate 1×1=1 row each. Therefore, the resulting table will have a total of 4+1+1=6 rows.

## 2. Bike Sharing

In [ ]:
trip = Table.read_table('./DS/trip.csv')
trip.show(3)

In [ ]:
a=trip.column(3)
items=np.unique(a, return_counts=True)
for k,v in zip(items[0],items[1]):
    print(k,v)

<br>

**Question 1** Create pivot table for Start Station and End Station. What value is being shown in each cell?

In [ ]:
pivotTable = trip.pivot('Start Station', 'End Station')
pivotTable
#The value shown in each cell of this table represents the count of bike rentals for trips originating at the corresponding 'Start Station' and ending at the corresponding 'End Station'.
#In other words, it is the aggregated number of trips between a specific starting point and a specific destination.

<br>

**Question 2** Create pivot table showing average duration of the trips between the  Start Station and End Station ?

In [ ]:
averagePivot = trip.pivot('Start Station', 'End Station', values='Duration', collect=np.mean)
averagePivot

## 3. San Francisco City Employee Salaries

This exercise is designed to give you practice with using the Table methods `.pivot` and `.group`. Here is a link to the [Python Reference](http://data8.org/sp24/reference/) in case you need a quick refresher. The [Table Function Visualizer](http://data8.org/interactive_table_functions/) may also be a helpful tool.


The data source we will use within this portion of the homework is [publicly provided](https://data.sfgov.org/City-Management-and-Ethics/Employee-Compensation/88g8-5mnd/data) by the City of San Francisco. We have filtered it to retain just the relevant columns and restricted the data to the calendar year 2019. Run the following cell to load our data into a table called `full_sf`.

In [ ]:
full_sf = Table.read_table("./DS/sf2019.csv")
full_sf.show(5)

The table has one row for each of the 44,525 San Francisco government employees in 2019.

The first four columns describe the employee's job. For example, the employee in the third row of the table had a job called "IS Business Analyst-Senior". We will call this the employee's *position* or *job title*. The job was in a Job Family called Information Systems (hence the IS in the job title), and was in the Adult Probation Department that is part of the Public Protection Organization Group of the government. You will mostly be working with the `Job` column.

The next three columns contain the dollar amounts paid to the employee in the calendar year 2019 for salary, overtime, and benefits. Note that an employee’s salary does not include their overtime earnings.

The last column contains the total compensation paid to the employee. It is the sum of the previous three columns:

$$\text{Total Compensation} = \text{Salary} + \text{Overtime} + \text{Benefits}$$

For this homework, we will be using the following columns:
1. `Organization Group`: A group of departments. For example, the Public Protection Org. Group includes departments such as the Police, Fire, Adult Protection, District Attorney, etc.
2. `Department`: The primary organizational unit used by the City and County of San Francisco.
3. `Job`: The specific position that a given worker fills.
4. `Total Compensation`: The sum of a worker's salary, overtime, and benefits in 2019.


Run the following cell to select the relevant columns and create a new table named `sf`.

In [ ]:
sf = full_sf.select("Job", "Department", "Organization Group",  "Total Compensation")
sf.show(5)

We want to use this table to generate arrays with the job titles of the members of each **Organization Group**.

**Question 1.** Set `job_titles` to a table with two columns. The first column should be called `Organization Group` and have the name of every "Organization Group" each listed only once in this column, and the second column should be called `Jobs` with each row in that second column containing an *array* of the names of all the job titles within that "Organization Group". Don't worry if there are multiple of the same job titles. **(9 Points)**

*Hint 1:* Think about how `group` works: it collects values into an array and then applies a function to that array. We have defined two functions below for you, and you will need to use one of them in your call to `group`.

*Hint 2:* You might need to rename one of the columns.


In [ ]:
# Pick one of the two functions defined below in your call to group.
def first_item(array):
    '''Returns the first item'''
    return array.item(0)

def full_array(array):
    '''Returns the array that is passed through'''
    return array

# Make a call to group using one of the functions above when you define job_titles
job_titles = sf.group("Organization Group", collect=full_array).relabeled("Job full_array", "Jobs").select("Organization Group","Jobs")
job_titles

<!-- BEGIN QUESTION -->

**Question 2.** At the moment, the `Job` column of the `sf` table is not sorted (no particular order). Would the arrays you generated in the `Jobs` column of the previous question be the same if we had sorted alphabetically instead before generating them? Explain your answer. To receive full credit, your answer should reference *how* the `.group` method works, and how sorting the `Jobs` column would affect this.  **(8 Points)**

*Note:* Two arrays are the **same** if they contain the same number of elements and the elements located at corresponding indexes in the two arrays are identical. An example of arrays that are NOT the same: `array([1,2]) != array([2,1])`.


The Table.group() method preserves the original row order when collecting values within a group. Therefore, if the sf table were sorted beforehand by the 'Job' column, the order in which .group places the job titles into the resulting arrays would change. Since the arrays must match not only in content but also in the order of their elements to be considered the same, the arrays generated before and after sorting would be different.

<!-- END QUESTION -->

**Question 3.** Set `department_ranges` to a table containing departments as the rows, and the organization groups as the columns. The values in the rows should correspond to a total compensation range, where range is defined as the **difference between the highest total compensation and the lowest total compensation in the department for that organization group**. **(9 Points)**

*Hint:* First you'll need to define a new function `compensation_range` which takes in an array of compensations and returns the range of compensations in that array.


In [ ]:
# Define compensation_range first
def compensation_range(array):
    '''Returns the range of compensations in the array'''
    return max(array) - min(array)

department_ranges = sf.pivot("Department", "Organization Group", values="Total Compensation", collect=compensation_range)
department_ranges

In [ ]:
# Define compensation_range first
def compensation_range(array):
    '''Returns the range of compensations in the array'''
    return max(array) - min(array)

<!-- BEGIN QUESTION -->

**Question 4.** Why might some of the row values be `0` in the `department_ranges` table from the previous question. **(8 Points)**


The values of 0 in the department_ranges table occur because the Total Compensation values for all employees within that specific combination of Organization Group and Department are identical. The compensation_range function calculates the difference between the maximum and minimum values in an array (max(array)−min(array)). Therefore, a return value of 0 indicates that the range of salaries for that particular group is 0, meaning there is no variation in the total compensation paid.

<!-- END QUESTION -->

**Question 5.** Find the number of departments appearing in the `sf` table that have an average total compensation of greater than 125,000 dollars; assign this value to the variable `num_over_125k`. **(9 Points)**

*Note:* The variable names provided are meant to help guide the intermediate steps and general thought process. Feel free to delete them if you'd prefer to start from scratch, but make sure your final answer is assigned to `num_over_125k`!


In [ ]:
depts_and_comp = sf.select("Department", "Total Compensation")
department_avg = depts_and_comp.group("Department", np.mean)
num_over_125k = department_avg.where("Total Compensation mean", are.above(125000)).num_rows
num_over_125k

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a pdf file for you to submit. **Please save before exporting!**

In [ ]:
# should change the directory and file name matching to yours
!jupyter nbconvert './DS/lab09(SSU).ipynb' --to pdf